# Probabilistic Decoding of Future Motor Intent with Mixture Density Networks

This notebook implements a PyTorch pipeline for decoding **future** hand velocity from motor cortical activity.
It includes:
- Sliding-window neural feature construction (PMd/M1 spike counts)
- Deterministic baseline decoder (MSE)
- Mixture Density Network decoder (multimodal uncertainty)
- Deterministic metrics: RMSE, MAE, R²
- Probabilistic metrics: NLL, calibration, prediction interval coverage

In [1]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.distributions import MultivariateNormal

torch.manual_seed(7)
np.random.seed(7)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

## 1) Data preparation helpers

Use `prepare_mc_maze_arrays` with MC_Maze arrays after loading from NLB. The helper expects neural bins and velocities already aligned in time.

In [2]:
def prepare_mc_maze_arrays(spikes_m1: np.ndarray, spikes_pmd: np.ndarray, velocity: np.ndarray):
    """
    Combine M1 and PMd binned spikes into one neural array.

    Parameters
    ----------
    spikes_m1 : [T, N_m1]
    spikes_pmd : [T, N_pmd]
    velocity : [T, 2] (vx, vy)
    """
    if not (len(spikes_m1) == len(spikes_pmd) == len(velocity)):
        raise ValueError("M1 spikes, PMd spikes, and velocity must share the same time dimension.")
    neural = np.concatenate([spikes_m1, spikes_pmd], axis=1).astype(np.float32)
    velocity = velocity.astype(np.float32)
    return neural, velocity


def build_windowed_features(neural: np.ndarray, velocity: np.ndarray, window_bins: int = 5, horizon_bins: int = 2):
    """
    Build temporally inclusive features and future-velocity targets.

    For each anchor time t, uses neural bins [t-window_bins+1, ..., t]
    and predicts velocity at t + horizon_bins.
    """
    if window_bins < 1 or horizon_bins < 1:
        raise ValueError("window_bins and horizon_bins must be positive integers.")

    T, n_neurons = neural.shape
    last_anchor = T - horizon_bins - 1
    first_anchor = window_bins - 1
    n_samples = last_anchor - first_anchor + 1
    if n_samples <= 0:
        raise ValueError("Not enough time bins for selected window/horizon.")

    X = np.zeros((n_samples, window_bins * n_neurons), dtype=np.float32)
    y = np.zeros((n_samples, 2), dtype=np.float32)

    out_idx = 0
    for t in range(first_anchor, last_anchor + 1):
        window = neural[t - window_bins + 1 : t + 1]  # [window_bins, n_neurons]
        X[out_idx] = window.reshape(-1)
        y[out_idx] = velocity[t + horizon_bins]
        out_idx += 1

    return X, y


def split_train_val_test(X: np.ndarray, y: np.ndarray, train_frac=0.7, val_frac=0.15):
    n = len(X)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)

    X_train, y_train = X[:n_train], y[:n_train]
    X_val, y_val = X[n_train:n_train + n_val], y[n_train:n_train + n_val]
    X_test, y_test = X[n_train + n_val:], y[n_train + n_val:]
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

## 2) Models

In [3]:
class BaselineDecoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256, output_dim: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)


class MDNDecoder(nn.Module):
    def __init__(self, input_dim: int, n_components: int = 5, hidden_dim: int = 256, out_dim: int = 2):
        super().__init__()
        self.n_components = n_components
        self.out_dim = out_dim
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.pi_head = nn.Linear(hidden_dim, n_components)
        self.mu_head = nn.Linear(hidden_dim, n_components * out_dim)
        # 2x2 lower-triangular covariance -> [l11, l21, l22]
        self.chol_head = nn.Linear(hidden_dim, n_components * 3)

    def forward(self, x):
        h = self.backbone(x)
        logits = self.pi_head(h)
        mu = self.mu_head(h).view(-1, self.n_components, self.out_dim)
        chol_raw = self.chol_head(h).view(-1, self.n_components, 3)

        L = torch.zeros(x.size(0), self.n_components, 2, 2, device=x.device)
        L[..., 0, 0] = F.softplus(chol_raw[..., 0]) + 1e-4
        L[..., 1, 0] = chol_raw[..., 1]
        L[..., 1, 1] = F.softplus(chol_raw[..., 2]) + 1e-4

        return logits, mu, L


def mdn_nll(logits, mu, L, y):
    # y: [B, 2]
    y_expanded = y.unsqueeze(1).expand_as(mu)
    mvn = MultivariateNormal(loc=mu, scale_tril=L)
    comp_log_prob = mvn.log_prob(y_expanded)  # [B, K]
    log_pi = F.log_softmax(logits, dim=-1)
    log_mix = torch.logsumexp(log_pi + comp_log_prob, dim=-1)
    return -log_mix.mean()

## 3) Training + evaluation utilities

In [1]:
def make_loader(X, y, batch_size=256, shuffle=False):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def train_baseline(model, train_loader, val_loader, epochs=20, lr=1e-3):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_state, best_val = None, float("inf")

    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            loss = F.mse_loss(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_losses.append(F.mse_loss(model(xb), yb).item())
        val = float(np.mean(val_losses))
        if val < best_val:
            best_val = val
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model


def train_mdn(model, train_loader, val_loader, epochs=20, lr=1e-3):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_state, best_val = None, float("inf")

    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits, mu, L = model(xb)
            loss = mdn_nll(logits, mu, L, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits, mu, L = model(xb)
                val_losses.append(mdn_nll(logits, mu, L, yb).item())
        val = float(np.mean(val_losses))
        if val < best_val:
            best_val = val
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0, keepdims=True)) ** 2)
    return float(1.0 - ss_res / (ss_tot + 1e-8))


def mdn_predictive_mean(logits, mu):
    pi = torch.softmax(logits, dim=-1).unsqueeze(-1)  # [B, K, 1]
    return (pi * mu).sum(dim=1)  # [B, 2]


def mdn_batch_nll(logits, mu, L, y):
    y_expanded = y.unsqueeze(1).expand_as(mu)
    mvn = MultivariateNormal(loc=mu, scale_tril=L)
    comp_log_prob = mvn.log_prob(y_expanded)
    log_pi = F.log_softmax(logits, dim=-1)
    return -(torch.logsumexp(log_pi + comp_log_prob, dim=-1))


def interval_coverage(y_true, lower, upper):
    inside = ((y_true >= lower) & (y_true <= upper)).all(axis=1)
    return float(np.mean(inside))


def mdn_sample(logits, mu, L, n_samples=200):
    """Draw Monte Carlo samples from mixture for each batch element."""
    B, K, _ = mu.shape
    pi = torch.softmax(logits, dim=-1)
    cat = torch.distributions.Categorical(pi)
    comp_idx = cat.sample((n_samples,))  # [S, B]

    samples = torch.zeros(n_samples, B, 2, device=mu.device)
    for b in range(B):
        means_b = mu[b]          # [K, 2]
        L_b = L[b]               # [K, 2, 2]
        idx_b = comp_idx[:, b]   # [S]
        z = torch.randn(n_samples, 2, device=mu.device)
        L_sel = L_b[idx_b]       # [S, 2, 2]
        m_sel = means_b[idx_b]   # [S, 2]
        samples[:, b] = m_sel + torch.einsum("sij,sj->si", L_sel, z)
    return samples

In [ ]:
## Download dataset and required packages if necessary
# !pip install git+https://github.com/neurallatents/nlb_tools.git
# !pip install dandi
# !dandi download https://gui.dandiarchive.org/#/dandiset/000128

from nlb_tools.nwb_interface import NWBDataset

## Load dataset
dataset = NWBDataset("sub-Jenkins/", "*train", split_heldout=False)

## View 'dataset.data'
dataset.data

## 4) Example run (synthetic data placeholder)

Replace the synthetic block with real MC_Maze arrays loaded from NLB.

In [ ]:
# Synthetic placeholder shaped like binned MC_Maze data
T = 6000
n_m1, n_pmd = 96, 96
spikes_m1 = np.random.poisson(0.15, size=(T, n_m1)).astype(np.float32)
spikes_pmd = np.random.poisson(0.12, size=(T, n_pmd)).astype(np.float32)

# Multimodal future intent toy signal
theta = np.linspace(0, 20 * np.pi, T)
base_v = np.stack([np.sin(theta), np.cos(theta)], axis=1).astype(np.float32)
branch = (np.random.rand(T, 1) > 0.5).astype(np.float32)
velocity = base_v + branch * np.array([[0.5, -0.5]], dtype=np.float32) + 0.1 * np.random.randn(T, 2).astype(np.float32)

neural, vel = prepare_mc_maze_arrays(spikes_m1, spikes_pmd, velocity)
X, y = build_windowed_features(neural, vel, window_bins=5, horizon_bins=2)

(train_X, train_y), (val_X, val_y), (test_X, test_y) = split_train_val_test(X, y)
train_loader = make_loader(train_X, train_y, batch_size=256, shuffle=True)
val_loader = make_loader(val_X, val_y, batch_size=512, shuffle=False)
test_loader = make_loader(test_X, test_y, batch_size=512, shuffle=False)

baseline = train_baseline(BaselineDecoder(train_X.shape[1]), train_loader, val_loader, epochs=12)
mdn = train_mdn(MDNDecoder(train_X.shape[1], n_components=5), train_loader, val_loader, epochs=12)

## 5) Deterministic and probabilistic evaluation

In [ ]:
baseline.eval()
mdn.eval()

all_y, all_baseline, all_mdn_mean, all_mdn_nll = [], [], [], []
all_q05, all_q95 = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        # baseline
        pred_b = baseline(xb)

        # mdn
        logits, mu, L = mdn(xb)
        pred_m = mdn_predictive_mean(logits, mu)
        nll_batch = mdn_batch_nll(logits, mu, L, yb)

        # sample-based prediction intervals and calibration proxy
        samples = mdn_sample(logits, mu, L, n_samples=300).cpu().numpy()  # [S, B, 2]
        q05 = np.quantile(samples, 0.05, axis=0)
        q95 = np.quantile(samples, 0.95, axis=0)

        all_y.append(yb.cpu().numpy())
        all_baseline.append(pred_b.cpu().numpy())
        all_mdn_mean.append(pred_m.cpu().numpy())
        all_mdn_nll.append(nll_batch.cpu().numpy())
        all_q05.append(q05)
        all_q95.append(q95)

Y = np.concatenate(all_y)
B = np.concatenate(all_baseline)
M = np.concatenate(all_mdn_mean)
NLL = np.concatenate(all_mdn_nll)
Q05 = np.concatenate(all_q05)
Q95 = np.concatenate(all_q95)

results = {
    "baseline_rmse": rmse(Y, B),
    "baseline_mae": mae(Y, B),
    "baseline_r2": r2(Y, B),
    "mdn_rmse_from_mean": rmse(Y, M),
    "mdn_mae_from_mean": mae(Y, M),
    "mdn_r2_from_mean": r2(Y, M),
    "mdn_nll": float(np.mean(NLL)),
    "mdn_90pct_interval_coverage": interval_coverage(Y, Q05, Q95),
}

results

## 6) Calibration summary

A simple calibration analysis compares empirical coverage to nominal coverage levels.

In [ ]:
def calibration_table(y_true, samples, levels=(0.5, 0.8, 0.9)):
    rows = []
    for level in levels:
        lo = (1 - level) / 2
        hi = 1 - lo
        q_lo = np.quantile(samples, lo, axis=0)
        q_hi = np.quantile(samples, hi, axis=0)
        coverage = interval_coverage(y_true, q_lo, q_hi)
        rows.append((level, coverage))
    return rows

# Recompute sample set over the test partition once for calibration table
all_samples = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(DEVICE)
        logits, mu, L = mdn(xb)
        all_samples.append(mdn_sample(logits, mu, L, n_samples=300).cpu().numpy())

samples_test = np.concatenate(all_samples, axis=1)  # [S, N, 2]
calibration = calibration_table(Y, samples_test, levels=(0.5, 0.8, 0.9))
calibration